In [ ]:
from importlib.util import find_spec

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
}

missing = [f"{module} (pip install {package})" for module, package in REQUIRED_PACKAGES.items() if find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        "Thiếu dependency để build embeddings/FAISS bằng notebook chính thức: " + ", ".join(missing)
    )


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())

KB_VERSION = os.getenv("HOMELAB_KB_VERSION", "v1")
RETRIEVER_VERSION = os.getenv("HOMELAB_RETRIEVER_VERSION", KB_VERSION)

ARTIFACTS_DIR = AI_LAB_ROOT / "artifacts" / f"retriever_{RETRIEVER_VERSION}"
REPORTS_DIR = AI_LAB_ROOT / "reports"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

KB_CHUNKS_PATH = ARTIFACTS_DIR / f"kb_chunks_{KB_VERSION}.json"
CHUNK_METADATA_PATH = ARTIFACTS_DIR / "chunk_metadata.json"

EMBEDDINGS_PATH = ARTIFACTS_DIR / "chunk_embeddings.npy"
FAISS_INDEX_PATH = ARTIFACTS_DIR / "faiss.index"
EMBEDDING_CONFIG_PATH = ARTIFACTS_DIR / "embedding_config.json"
RETRIEVER_MANIFEST_PATH = ARTIFACTS_DIR / "retriever_manifest.json"

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("KB_CHUNKS_PATH =", KB_CHUNKS_PATH)

In [ ]:
with open(KB_CHUNKS_PATH, "r", encoding="utf-8") as f:
    kb_chunks = json.load(f)

with open(CHUNK_METADATA_PATH, "r", encoding="utf-8") as f:
    chunk_metadata = json.load(f)

print("Số chunks:", len(kb_chunks))
pd.DataFrame(kb_chunks)[["chunk_id", "title", "section", "risk_level"]]

In [ ]:
def build_passage_text(chunk):
    return "passage: " + chunk["chunk_text"].strip()

chunk_ids = [c["chunk_id"] for c in kb_chunks]
texts = [build_passage_text(c) for c in kb_chunks]

print("Ví dụ chunk text đầu tiên:\n")
print(texts[0][:1200])

In [ ]:
MODEL_NAME = os.getenv("HOMELAB_EMBED_MODEL", "intfloat/multilingual-e5-small")
model = SentenceTransformer(MODEL_NAME)
print("Loaded model:", MODEL_NAME)


In [ ]:
embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

In [ ]:
np.save(EMBEDDINGS_PATH, embeddings)
print("Đã ghi:", EMBEDDINGS_PATH)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype("float32"))

faiss.write_index(index, str(FAISS_INDEX_PATH))
print("Đã ghi:", FAISS_INDEX_PATH)
print("Tổng số vector trong index:", index.ntotal)

In [ ]:
embedding_config = {
    "model_name": MODEL_NAME,
    "embedding_dimension": int(embeddings.shape[1]),
    "normalized": True,
    "index_type": "IndexFlatIP",
    "text_field": "chunk_text",
    "passage_prefix": "passage: ",
    "query_prefix": "query: "
}

with open(EMBEDDING_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(embedding_config, f, ensure_ascii=False, indent=2)

print("Đã ghi:", EMBEDDING_CONFIG_PATH)

In [ ]:
retriever_manifest = {
    "retriever_version": RETRIEVER_VERSION,
    "kb_version": KB_VERSION,
    "artifact_dir": f"ai_lab/artifacts/retriever_{RETRIEVER_VERSION}",
    "kb_file": f"kb_chunks_{KB_VERSION}.json",
    "metadata_file": "chunk_metadata.json",
    "embeddings_file": "chunk_embeddings.npy",
    "faiss_index_file": "faiss.index",
    "embedding_config_file": "embedding_config.json",
    "chunk_count": len(kb_chunks),
    "model_name": MODEL_NAME,
    "top_k_default": 3,
    "build_route": "official_notebook",
}

with open(RETRIEVER_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(retriever_manifest, f, ensure_ascii=False, indent=2)

print("Đã ghi:", RETRIEVER_MANIFEST_PATH)


In [ ]:
def search(query, top_k=3):
    query_text = "query: " + query.strip()
    q_emb = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = kb_chunks[idx]
        results.append({
            "rank": len(results) + 1,
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "section": chunk["section"],
            "source_id": chunk["source_id"],
            "content_preview": chunk["content"][:300]
        })
    return results

if KB_VERSION == "v1":
    test_queries = [
        "xét nghiệm máu là gì",
        "khi nào đau ngực cần đi cấp cứu",
        "khó thở khi nào nguy hiểm",
        "dấu hiệu nhiễm trùng nặng cần đi viện",
    ]
else:
    test_queries = [
        "troponin là xét nghiệm gì",
        "d-dimer dùng để làm gì",
        "đau đầu dữ dội đột ngột có cần cấp cứu không",
        "đau bụng dữ dội kèm nôn ra máu có nguy hiểm không",
    ]

for q in test_queries:
    print("=" * 100)
    print("QUERY:", q)
    results = search(q, top_k=3)
    for r in results:
        print(f"[{r['rank']}] score={r['score']:.4f} | {r['title']} | {r['section']} | {r['source_id']}")
    print()


In [ ]:
rows = []
for q in test_queries:
    results = search(q, top_k=3)
    for r in results:
        rows.append({
            "query": q,
            "rank": r["rank"],
            "score": r["score"],
            "chunk_id": r["chunk_id"],
            "title": r["title"],
            "section": r["section"],
            "source_id": r["source_id"]
        })

df_smoke = pd.DataFrame(rows)
df_smoke